# 03 — Entrenamiento completo en Google Colab Pro (funcionalidad 010, T28 a T33)

Corre el pipeline completo con la **receta v1** y **saltea las etapas ya terminadas**: si Colab corta la sesión, se vuelve a ejecutar todo y sigue desde donde quedó.

```
obtener → validar → preparar → entrenar → exportar → evaluar → publicar
```

| Si se corta durante… | Al volver a ejecutar todo… |
|---|---|
| obtener o preparar | Se repite la descarga (las imágenes crudas no se guardan en Drive) |
| entrenar | Se recuperan los datos preparados de Drive y el entrenamiento **sigue desde la última época** |
| exportar, evaluar o publicar | Se recupera todo lo anterior y solo se hace lo que falta |

Una etapa cuenta como terminada solo si su resultado corresponde **al mismo manifiesto** (misma huella): nunca se mezclan resultados de recetas distintas.

**Antes de ejecutar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo* → **T4** (alcanza) o **L4** (más rápida).
2. Secretos `HF_TOKEN` y `GITHUB_TOKEN` con **Acceso al cuaderno**. Para publicar en Hugging Face (T33), el `HF_TOKEN` tiene que ser de tipo **Write**.
3. Que la computadora no se suspenda y tocar la página cada 30 a 40 minutos.
4. *Entorno de ejecución → Ejecutar todas*.

## 1. Parámetros

In [ ]:
from pathlib import Path

# Rama o commit. Para el entrenamiento que se publica, usar un commit (40 caracteres).
REVISION = "funcionalidad/010-red-neuronal"
CONFIGURACION_BASE = "configuracion/v1.yaml"
VERSION = "1.0.0"

# Conjunto propio en Drive (carpeta privada). Si no existe todavía, se evalúa sin él.
CONJUNTO_PROPIO = Path("/content/drive/MyDrive/evidentia/conjunto-propio")

# Dónde publicar: registro local en Drive, o el repositorio privado de Hugging Face (E03).
PUBLICAR_EN_HUGGING_FACE = False
REPOSITORIO_MODELOS = "valentinasalmon/pfg-clasificador-sintetico"

REPOSITORIO = "valentinasalmon/PFG-sistema-deteccion-IA"
from google.colab import drive

drive.mount("/content/drive")
TRABAJO = Path("/content")
CODIGO = TRABAJO / "codigo"

## 2. Credenciales y código

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
_token_github = userdata.get("GITHUB_TOKEN")
if not os.environ["HF_TOKEN"] or not _token_github:
    raise RuntimeError("Falta HF_TOKEN o GITHUB_TOKEN en los secretos (ícono de la llave), "
                       "o están vacíos. Cargalos con Acceso al cuaderno activado y volvé a ejecutar todo.")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
print("Credenciales cargadas (no se muestran).")

import base64
import subprocess


def git_seguro(*argumentos, token=None):
    """Ejecuta git sin que el token aparezca nunca en el comando ni en los errores.

    El token viaja en una variable de entorno de configuración de git
    (cabecera de autorización), no en la URL. Si git falla, se muestra solo
    su mensaje de error, que no contiene el token.
    """
    entorno = dict(os.environ)
    if token:
        credencial = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        entorno.update({
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credencial}",
            "GIT_TERMINAL_PROMPT": "0",
        })
    proceso = subprocess.run(["git", *argumentos], env=entorno, capture_output=True, text=True)
    if proceso.returncode != 0:
        mensaje = proceso.stderr.replace(token, "***") if token else proceso.stderr
        if "could not read Username" in mensaje or "Authentication failed" in mensaje:
            mensaje += (
                "\n\nGitHub rechazó el GITHUB_TOKEN: venció, está mal copiado o no tiene acceso "
                "a este repositorio. Revisá en GitHub (Settings → Developer settings → "
                "Fine-grained tokens) que tenga el repositorio PFG-sistema-deteccion-IA y "
                "Contents: Read-only; si dudás, creá uno nuevo, reemplazá el secreto y "
                "ejecutá todo otra vez."
            )
        raise RuntimeError(f"git {argumentos[0]} falló:\n{mensaje.strip()}") from None
    return proceso.stdout.strip()


if CODIGO.exists():
    subprocess.run(["rm", "-rf", str(CODIGO)], check=True)
git_seguro("clone", "--quiet", f"https://github.com/{REPOSITORIO}.git", str(CODIGO),
           token=_token_github)
git_seguro("-C", str(CODIGO), "fetch", "--quiet", "origin", REVISION, token=_token_github)
git_seguro("-C", str(CODIGO), "checkout", "--quiet", REVISION)
del _token_github

commit = git_seguro("-C", str(CODIGO), "rev-parse", "HEAD")
print("Commit en uso:", commit)

## 3. Configuración efectiva y recuperación desde Drive

El respaldo vive en `MyDrive/evidentia/<nombre de la receta>/`. Si ahí hay una configuración **distinta** de la actual, el cuaderno se detiene: para una receta nueva hay que cambiarle el `nombre` en el YAML (así cada receta tiene su carpeta).

In [ ]:
import shutil
import tarfile
import yaml

base = yaml.safe_load((CODIGO / "red_neuronal" / CONFIGURACION_BASE).read_text())
base["directorio_trabajo"] = str(TRABAJO / "trabajo")
texto_configuracion = yaml.safe_dump(base, allow_unicode=True, sort_keys=False)
CONFIGURACION = TRABAJO / "configuracion_efectiva.yaml"
CONFIGURACION.write_text(texto_configuracion)

RESPALDO = Path("/content/drive/MyDrive/evidentia") / base["nombre"]
DATOS_PREPARADOS = Path("/content/drive/MyDrive/evidentia/datos") / base["nombre"] / "preparado.tar"
RESPALDO.mkdir(parents=True, exist_ok=True)
anterior = RESPALDO / "configuracion_efectiva.yaml"
if anterior.exists() and anterior.read_text() != texto_configuracion:
    raise RuntimeError(
        f"En {RESPALDO} hay resultados de otra configuración. Si cambiaste la receta, "
        "cambiale el 'nombre' en el YAML (por ejemplo, v1b) o borrá esa carpeta de Drive."
    )
anterior.write_text(texto_configuracion)

def respaldar():
    """Copia a Drive la carpeta de trabajo y el registro, sin imágenes."""
    for carpeta in ("trabajo", "registro"):
        origen = TRABAJO / carpeta
        if origen.exists():
            shutil.copytree(origen, RESPALDO / carpeta, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("crudo", "preparado", "propio"))
    manifiesto = TRABAJO / "trabajo/preparado/manifiesto.csv"
    if manifiesto.exists():
        (RESPALDO / "trabajo/preparado").mkdir(parents=True, exist_ok=True)
        shutil.copy2(manifiesto, RESPALDO / "trabajo/preparado/manifiesto.csv")

def restaurar():
    """Trae de Drive lo ya hecho y, si existen, los datos preparados."""
    for carpeta in ("trabajo", "registro"):
        if (RESPALDO / carpeta).exists():
            shutil.copytree(RESPALDO / carpeta, TRABAJO / carpeta, dirs_exist_ok=True)
            print("Recuperado de Drive:", carpeta)
    if DATOS_PREPARADOS.exists() and not (TRABAJO / "trabajo/preparado/real").exists():
        print("Extrayendo los datos preparados de Drive (unos minutos)...")
        with tarfile.open(DATOS_PREPARADOS) as paquete:
            paquete.extractall(TRABAJO / "trabajo", filter="data")

restaurar()
print(texto_configuracion)

## 4. Qué etapas ya están hechas

In [ ]:
import subprocess, time

def correr(*argumentos):
    """Ejecuta el paquete y muestra su salida línea por línea (Colab no la muestra sola)."""
    proceso = subprocess.Popen(
        ["python", "-m", "clasificador", *argumentos],
        cwd=CODIGO / "red_neuronal", text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for linea in proceso.stdout:
        print(linea, end="")
    return proceso.wait()

correr("estado", "--configuracion", str(CONFIGURACION))

## 5. Ejecutar solo lo que falta

Cada etapa terminada se respalda en Drive. Después de `preparar` se guardan **los datos preparados comprimidos** en `MyDrive/evidentia/datos/<receta>/preparado.tar` (T28): así un corte durante el entrenamiento no obliga a bajar todo de nuevo.

In [ ]:
import json
import sys
sys.path.insert(0, str(CODIGO / "red_neuronal"))  # el kernel ya estaba abierto al instalar
from clasificador import pipeline
from clasificador.configuracion import cargar_configuracion

tiempos = {}

def estado():
    return pipeline.estado_etapas(cargar_configuracion(CONFIGURACION))

def ejecutar_etapa(etapa, *extra):
    """Ejecuta una etapa salvo que ya esté terminada; se detiene si falla."""
    if estado()[etapa] and not (etapa == "evaluar" and falta_conjunto_propio()):
        print(f"--> {etapa}: ya estaba hecha, se saltea")
        return
    inicio = time.perf_counter()
    codigo_salida = correr(etapa, "--configuracion", str(CONFIGURACION), *extra)
    tiempos[etapa] = round(time.perf_counter() - inicio, 1)
    print(f"--> {etapa}: {tiempos[etapa]} s")
    if codigo_salida != 0:
        respaldar()
        raise RuntimeError(f"La etapa '{etapa}' falló: el motivo está en las líneas de arriba.")
    respaldar()

def falta_conjunto_propio():
    """El conjunto propio existe pero todavía no se evaluó con este modelo."""
    metricas = TRABAJO / "trabajo/evaluacion/condiciones_reales/metricas.json"
    return CONJUNTO_PROPIO.exists() and not metricas.exists()

def guardar_datos_preparados():
    """Comprime los datos preparados en Drive, una sola vez por receta."""
    if DATOS_PREPARADOS.exists():
        return
    DATOS_PREPARADOS.parent.mkdir(parents=True, exist_ok=True)
    temporal = TRABAJO / "preparado.tar"
    print("Guardando los datos preparados en Drive (unos minutos)...")
    with tarfile.open(temporal, "w") as paquete:  # JPEG ya comprimido: sin gzip
        paquete.add(TRABAJO / "trabajo/preparado", arcname="preparado")
    shutil.copy2(temporal, DATOS_PREPARADOS)
    temporal.unlink()
    print("Datos preparados en:", DATOS_PREPARADOS)

ejecutar_etapa("obtener")
ejecutar_etapa("validar")
ejecutar_etapa("preparar")
guardar_datos_preparados()

In [ ]:
ejecutar_etapa("entrenar")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
ejecutar_etapa("exportar", "--version", VERSION)
extra = ["--conjunto-propio", str(CONJUNTO_PROPIO)] if CONJUNTO_PROPIO.exists() else []
print("Conjunto propio:", CONJUNTO_PROPIO if extra else "todavía no está en Drive")
ejecutar_etapa("evaluar", *extra)

### Publicación a través de la compuerta

La compuerta exige AUC de prueba ≥ 0,90, ONNX fiel, ficha completa, huella del manifiesto y **código confirmado** (por eso conviene usar un commit en `REVISION`). Si rechaza, muestra **todos** los motivos: copiarlos al chat. El rechazo no borra nada.

In [ ]:
registro = (f"hugging_face:{REPOSITORIO_MODELOS}" if PUBLICAR_EN_HUGGING_FACE
            else f"local:{TRABAJO / 'registro'}")
print("Registro:", registro)
try:
    ejecutar_etapa("publicar", "--version", VERSION, "--registro", registro)
except RuntimeError as error:
    print(error)
    compuerta = TRABAJO / "trabajo/compuerta.json"
    if compuerta.exists():
        print(json.dumps(json.loads(compuerta.read_text()), indent=2, ensure_ascii=False))

## 6. Resultados para la bitácora y el capítulo 11

In [ ]:
import shutil
trabajo = TRABAJO / "trabajo"
leer = lambda ruta: json.loads((trabajo / ruta).read_text()) if (trabajo / ruta).exists() else None
entrenamiento = leer("entrenamiento/resultado.json") or {}
exportacion = leer("exportacion/exportacion.json") or {}
prueba = leer("evaluacion/prueba/metricas.json") or {}
no_visto = leer("evaluacion/no_visto/metricas.json")
reales = leer("evaluacion/condiciones_reales/metricas.json")
uso_disco = shutil.disk_usage(TRABAJO)
resumen = {
    "commit": exportacion.get("commit"),
    "receta": base["nombre"],
    "tiempos_segundos": tiempos or "etapas recuperadas de una sesión anterior",
    "mejor_auc_validacion": entrenamiento.get("mejor_auc"),
    "epoca_mejor": entrenamiento.get("epoca_mejor"),
    "auc_por_epoca": [round(h.get("auc_validacion", 0), 4) for h in entrenamiento.get("historial", [])],
    "temperatura": entrenamiento.get("temperatura"),
    "ece_antes_despues": [entrenamiento.get("ece_antes"), entrenamiento.get("ece_despues")],
    "auc_prueba": prueba.get("total", {}).get("auc"),
    "f1_prueba": prueba.get("total", {}).get("f1"),
    "auc_no_visto": (no_visto or {}).get("total", {}).get("auc"),
    "auc_condiciones_reales": (reales or {}).get("total", {}).get("auc"),
    "por_variante": {k: v.get("auc") for k, v in (reales or {}).get("por_variante", {}).items()},
    "por_escena": {k: v.get("auc") for k, v in (reales or {}).get("por_escena", {}).items()},
    "por_herramienta": {k: v.get("auc") for k, v in (reales or {}).get("por_generador", {}).items()},
    "diferencia_onnx": exportacion.get("diferencia_maxima"),
    "latencia_ms_cpu": exportacion.get("latencia_ms_por_imagen"),
    "disco_usado_gb": round((uso_disco.total - uso_disco.free) / 1e9, 1),
}
(RESPALDO / "resumen.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False))
print(json.dumps(resumen, indent=2, ensure_ascii=False))

In [ ]:
from IPython.display import Image as MostrarImagen, Markdown, display
for conjunto in ("prueba", "no_visto", "condiciones_reales"):
    for grafico in ("curva_roc.png", "matriz_confusion.png"):
        ruta = trabajo / "evaluacion" / conjunto / grafico
        if ruta.exists():
            print(conjunto, "·", grafico)
            display(MostrarImagen(str(ruta)))
fichas = sorted((TRABAJO / "registro").glob(f"v{VERSION}/ficha.md"))
display(Markdown(fichas[0].read_text()) if fichas else Markdown("*No hay ficha publicada todavía.*"))

## 7. Qué pasar al chat

1. El **resumen** de la sección 6 (entre llaves).
2. Si la compuerta rechazó, sus **motivos**.
3. Capturas de las curvas ROC y matrices de confusión.
4. Qué GPU se usó y si se cortó alguna vez.

Antes de pegar, revisar que no aparezca ningún token (`hf_…`, `github_pat_…`). **No** se suben al repositorio ni las imágenes ni el modelo: quedan en `MyDrive/evidentia/`.